# Model evaluation: Swin-YNet vs EfficientNet-B2 burned-area detection (T29TPG)

This notebook is the quantitative and qualitative comparison of the two burned-area
change-detection models run through the shared `inference/` pipeline, evaluated
against ICNF's official 2025 fire perimeters. Both runs use the identical
before/after scene pair (before 2025-07-07, after 2025-10-15), so the comparison
is on the same pixel grid and the same prediction task.

The evaluation design (which metrics, why, and what is deliberately excluded) is
not improvised here. It follows `documents/evaluation_protocol.md`, the project's
own methodology document, and is grounded in the burned-area accuracy assessment
literature. The key decisions, briefly:

- **MCC (Matthews Correlation Coefficient) is the headline metric**, not overall
  accuracy or F1 alone (Matthews, 1975; Chicco & Jurman, 2020).
- **Precision, recall and F1 are reported on the burned class only** (Raschka,
  Liu & Mirjalili, 2022; Roteta et al., 2019).
- **ROC/AUC is excluded**: neither adapter exposes a probability surface at the
  public interface, only a hard 0/1 label, so there is no threshold to sweep
  (Fawcett, 2006).
- **Ground truth is the ICNF polygon set, eroded by one pixel (10 m)** before
  rasterisation, to remove boundary mixed-pixel ambiguity (Stehman & Foody, 2019).
- **Only fires that started after the before-date and ended before the after-date**
  are scored, so a fire the model never had a chance to see is not held against it.
- **Fire-size-stratified (object-based) accuracy is reported alongside the pixel
  metrics**, because the ICNF fire-size distribution in this tile is extremely
  right-skewed (182 events, median 1.98 ha, mean 70.1 ha) and a 1.98 ha fire is a
  handful of pixels inside a 655 ha chip, too small for per-pixel metrics alone to
  be informative (Roteta et al., 2019).

Where a result has a direct implication for the post-processing hyperparameters
each model already ships with (`MIN_PATCH_SIZE`, `CLOSING_RADIUS`, `CUTS_THRESHOLD`),
that implication is stated explicitly rather than left for the reader to infer.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
from scipy.stats import chi2 as chi2_dist
from sklearn.metrics import (
    confusion_matrix, matthews_corrcoef, precision_score, recall_score, f1_score,
)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

for _c in (Path.cwd(), Path.cwd().parent):
    if (_c / "utils" / "config.py").exists():
        sys.path.insert(0, str(_c))
        break

from utils.config import load_config

cfg = load_config()
print("repository root:", cfg.repo_root)


## 2. Locate the two models' full-tile outputs

Both runs were written under `outputs/predictions/hannah/` using the pipeline's
`--out` flag, a personal subfolder convention adopted so that two people running
the same model/date pair on a shared OneDrive folder do not silently overwrite
each other's GeoTIFFs (the manifest and filename stem carry no run-id or
username, only `tile_model_before_after`).

`find_raster` searches recursively under `predictions_dir`, so it finds outputs
regardless of which personal subfolder they were written to, and matches the
model kind explicitly (`efficientnet_b2` vs `swin_ynet`) since both now coexist
in the same tree.


In [ ]:
BEFORE = "2025-07-07"
AFTER  = "2025-10-15"

PRED_DIR = cfg.predictions_dir
LABELS   = cfg.repo_root / "data/processed/icnf_burned_labels_t29tpg_2025.tif"
ICNF_SHP = cfg.repo_root / "data/shapefiles/ground_truth_ICNF/ardida_2025.shp"

PIX_HA = 0.01   # one 10 m x 10 m pixel = 0.01 ha

MODELS = {
    "efficientnet_b2": "EfficientNet-B2",
    "swin_ynet":        "Swin-YNet",
}

def find_raster(model_kind, after_date, kind):
    # Locate the full-tile raster for a given model and after-date.
    a = after_date.replace("-", "")
    b = BEFORE.replace("-", "")
    hits = sorted(PRED_DIR.rglob(f"T29TPG_{model_kind}_{b}_{a}_{kind}.tif"))
    if not hits:
        raise FileNotFoundError(
            f"No '{kind}' map for model={model_kind}, after={after_date} under {PRED_DIR}. "
            f"Run the inference first, e.g.\n"
            f"  python -m inference.run --model-kind {model_kind} --device cpu "
            f"--before-date {BEFORE} --after-date {after_date} --out ./outputs/predictions/<you>"
        )
    if len(hits) == 1:
        return hits[0]
    # Several people may have run the same model/date pair; keep the most complete map.
    def coverage(p):
        with rasterio.open(p) as r:
            return int((r.read(1) != 255).sum())
    return max(hits, key=coverage)

def read(path):
    with rasterio.open(path) as r:
        return r.read(1)

burned = {m: read(find_raster(m, AFTER, "burned"))   for m in MODELS}
obs    = {m: read(find_raster(m, AFTER, "observed")) for m in MODELS}

for m in MODELS:
    print(f"{MODELS[m]:>16}: burned map {burned[m].shape}, observed coverage "
          f"{(obs[m] == 1).sum() * PIX_HA:,.0f} ha")


## 3. Grid alignment check

Both models' adapters write to the chip grid reconstructed from the HDF5 cube
(`inference/chips.py`), which is also the grid the project's static ICNF label
raster was built on (`run.py`'s own `_check_grid_alignment` does this same check
at inference time). Confirming it again here means every later pixel-by-pixel
comparison is comparing the same physical pixel, not a resampled approximation.


In [ ]:
with rasterio.open(LABELS) as r:
    GRID_TRANSFORM = r.transform
    GRID_SHAPE     = (r.height, r.width)
    EXTENT         = (r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top)
    labels = r.read(1)

scored = labels != 255   # study-area footprint (the part of the tile inside Portugal)

for m in MODELS:
    assert burned[m].shape == GRID_SHAPE, f"{m}: grid mismatch {burned[m].shape} vs {GRID_SHAPE}"
print("grid OK:", GRID_SHAPE, "study-area pixels:", scored.sum(), f"({scored.sum() * PIX_HA:,.0f} ha)")


## 4. Ground truth: eroded, date-windowed ICNF fires

Each fire polygon is shrunk inward by one pixel (10 m) before rasterisation.
Pixels straddling a fire's true perimeter are a physical mix of burned and
unburned land cover at 10 m resolution; giving that ring a hard 0/1 label adds
label noise that penalises both models equally but for a reason that has nothing
to do with model quality. This is standard practice in land-cover accuracy
assessment (Stehman & Foody, 2019) and is exactly what
`notebooks/burned_area_date_comparison.ipynb` already implements; the function
below is unchanged from that notebook.

Fires are also restricted to the before/after window: a fire that had already
happened before the before-date is already a burn scar in the baseline (the
model cannot detect a change that occurred before its own reference image), and
a fire after the after-date has not happened yet at the time of the after image.


In [ ]:
fires = gpd.read_file(ICNF_SHP).to_crs("EPSG:32629")
fires = fires.cx[EXTENT[0]:EXTENT[1], EXTENT[2]:EXTENT[3]].copy()
fires["start"] = pd.to_datetime(fires["DH_Inicio"])
fires["end"]   = pd.to_datetime(fires["DH_Fim"])

def truth_mask(before_date, after_date, erode_m=10):
    sel = fires[(fires["start"] >= pd.Timestamp(before_date)) &
                (fires["end"]   <= pd.Timestamp(after_date))]
    geoms = [g for g in sel.geometry.buffer(-erode_m) if not g.is_empty]
    if not geoms:
        return np.zeros(GRID_SHAPE, dtype=bool)
    arr = rasterize([(g, 1) for g in geoms], out_shape=GRID_SHAPE,
                     transform=GRID_TRANSFORM, fill=0, dtype="uint8")
    return arr.astype(bool)

REAL = truth_mask(BEFORE, AFTER)
fires_in_window = fires[(fires["start"] >= pd.Timestamp(BEFORE)) & (fires["end"] <= pd.Timestamp(AFTER))]
print(f"ground truth {BEFORE} -> {AFTER}: {len(fires_in_window)} fires, {REAL.sum() * PIX_HA:,.0f} ha (after erosion)")


## 5. Pixel-based metrics

Computed on the burned class only, restricted to pixels both images actually
observed (`observed == 1`) and inside the study-area footprint. Both models'
`observed` masks come from the same chip-reading step (`inference/chips.py`)
applied to the same before/after scenes, so they are pixel-for-pixel identical;
the comparison below is therefore over exactly the same evaluation set for both
models, which is what makes a head-to-head difference meaningful rather than an
artefact of one model being scored on an easier subset of the tile.

**MCC** is the headline number. It uses all four confusion-matrix cells and,
unlike F1, is symmetric under swapping which class is called "positive": this
matters here because the burned class is the small minority class by a wide
margin (about 1.7% of the study area at the upper end, see the ground-truth area
above against the labelled tile total). Chicco & Jurman (2020) show F1 can be
inflated under exactly this kind of imbalance because it ignores true negatives
entirely; MCC cannot be inflated the same way.

```
MCC = (TP*TN - FP*FN) / sqrt((TP+FP)(TP+FN)(TN+FP)(TN+FN))
```


In [ ]:
def pixel_metrics(model_kind):
    mask = (obs[model_kind] == 1) & scored
    y_true = REAL[mask].astype(int)
    y_pred = (burned[model_kind][mask] == 1).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model":     MODELS[model_kind],
        "mcc":       matthews_corrcoef(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall":    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1":        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "n_evaluated": int(mask.sum()),
    }

results = pd.DataFrame([pixel_metrics(m) for m in MODELS]).set_index("model")
results[["mcc", "precision", "recall", "f1"]] = results[["mcc", "precision", "recall", "f1"]].round(4)
results


### 5.1 Headline comparison (interactive)

In [ ]:
metrics_long = results.reset_index().melt(
    id_vars="model", value_vars=["mcc", "precision", "recall", "f1"],
    var_name="metric", value_name="value",
)
fig = px.bar(
    metrics_long, x="metric", y="value", color="model", barmode="group",
    text="value", range_y=(0, 1),
    title="Pixel-based metrics by model (burned class, observed & in-study-area pixels only)",
    color_discrete_map={"EfficientNet-B2": "#2c7fb8", "Swin-YNet": "#d95f0e"},
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(yaxis_title="score", xaxis_title="", legend_title="")
fig.show()


### 5.2 Confusion matrices (normalised by row)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(MODELS.values()))
for i, m in enumerate(MODELS, start=1):
    r = results.loc[MODELS[m]]
    cm = np.array([[r.tn, r.fp], [r.fn, r.tp]], dtype=float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig.add_trace(
        go.Heatmap(
            z=cm_norm, x=["pred: not burned", "pred: burned"], y=["true: not burned", "true: burned"],
            text=cm.astype(int), texttemplate="%{text:,}", colorscale="Blues", zmin=0, zmax=1,
            showscale=(i == 2),
        ),
        row=1, col=i,
    )
fig.update_layout(title="Confusion matrices (cell text = pixel count, colour = row-normalised proportion)")
fig.show()


## 6. McNemar's test: are the two models' errors actually different?

A point-estimate comparison (model A's MCC is higher than model B's) does not
by itself say whether the difference reflects a systematic difference between
the models or noise in which pixels happened to be wrong. McNemar's test is the
standard way to compare two classifiers on the **same** paired observations: it
looks only at the pixels where the two models disagree (one right, one wrong)
and tests whether that disagreement is symmetric.

```
b = pixels where A correct, B wrong
c = pixels where A wrong,   B correct
chi2 = (|b - c| - 1)^2 / (b + c)      (continuity-corrected)
```

**Caveat that matters more than the test itself here**: the evaluation set is
tens of millions of pixels. At that sample size, McNemar's test (like almost any
significance test) will return an overwhelmingly significant p-value even for a
practically tiny difference, because statistical significance is partly a
function of sample size, not just effect size. The number worth reporting
alongside the p-value is therefore the **discordant pixel count and which model
wins on it**, not the p-value alone.


In [ ]:
def correctness(model_kind):
    mask = (obs[model_kind] == 1) & scored
    y_true = REAL[mask]
    y_pred = burned[model_kind][mask] == 1
    return (y_true == y_pred), mask

model_a, model_b = list(MODELS)
correct_a, mask_a = correctness(model_a)
correct_b, mask_b = correctness(model_b)
assert (mask_a == mask_b).all(), "models were not evaluated on the same pixel set"

b = int(( correct_a & ~correct_b).sum())   # A right, B wrong
c = int((~correct_a &  correct_b).sum())   # A wrong, B right
both_right = int((correct_a & correct_b).sum())
both_wrong = int((~correct_a & ~correct_b).sum())

chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
p_value = 1 - chi2_dist.cdf(chi2_stat, df=1)

print(f"both correct:                 {both_right:,}")
print(f"both wrong:                    {both_wrong:,}")
print(f"{MODELS[model_a]} right, {MODELS[model_b]} wrong:  {b:,}")
print(f"{MODELS[model_a]} wrong, {MODELS[model_b]} right:  {c:,}")
print(f"McNemar chi2 = {chi2_stat:,.1f}, p = {p_value:.3g}")
print(f"-> {MODELS[model_a] if b > c else MODELS[model_b]} is right on more of the disputed pixels "
      f"({max(b, c):,} vs {min(b, c):,}, {100 * max(b, c) / (b + c):.1f}% of disagreements)")


## 7. Fire-size-stratified (object-based) accuracy

Per-pixel metrics treat every burned pixel as an independent sample, but the
ICNF fire-size distribution in T29TPG is extremely skewed: 182 events, median
1.98 ha, mean 70.1 ha, max 4,196.6 ha. A 256x256 chip at 10 m covers 655 ha, so
a median-sized fire is roughly 0.3% of one chip, a few pixels. Per-pixel recall
on such an event is close to meaningless noise, and MCC/F1 are undefined when
there are zero positive predictions in the window being measured.

Following `evaluation_protocol.md` (and Roteta et al., 2019, on minimum mapping
units for Sentinel-2 burned-area products), fires below **65 ha** (about 10% of
one chip's area) are excluded from this stratified analysis, and the **4 events
above 945 ha** are the headline evaluation set. This section instead asks an
object-level question per evaluable fire: *what fraction of this fire's eroded
area did each model actually flag as burned?* That is a per-event recall, and
plotting it against fire size directly tests whether detection rate depends on
event size, which is exactly the question that should drive any retuning of
`MIN_PATCH_SIZE` (the post-processing step that deletes small connected
components).


In [ ]:
CHIP_AREA_HA = (256 * 10) ** 2 / 1e4
SIZE_THRESHOLD_HA = CHIP_AREA_HA * 0.10   # ~65 ha
HEADLINE_THRESHOLD_HA = 945

evaluable = fires_in_window[fires_in_window["AreaHaSIG"] > SIZE_THRESHOLD_HA].copy()
evaluable = evaluable.sort_values("AreaHaSIG", ascending=False).reset_index(drop=True)
print(f"evaluable fires (>{SIZE_THRESHOLD_HA:.0f} ha): {len(evaluable)}  "
      f"(headline >{HEADLINE_THRESHOLD_HA} ha: {(evaluable['AreaHaSIG'] > HEADLINE_THRESHOLD_HA).sum()})")

def per_fire_recall(model_kind, geom, erode_m=10):
    eroded = geom.buffer(-erode_m)
    if eroded.is_empty:
        return np.nan, 0
    fire_mask = rasterize([(eroded, 1)], out_shape=GRID_SHAPE, transform=GRID_TRANSFORM,
                           fill=0, dtype="uint8").astype(bool)
    fire_mask &= (obs[model_kind] == 1) & scored
    n = int(fire_mask.sum())
    if n == 0:
        return np.nan, 0
    hit = int((burned[model_kind][fire_mask] == 1).sum())
    return hit / n, n

rows = []
for _, fire in evaluable.iterrows():
    for m in MODELS:
        recall, n_px = per_fire_recall(m, fire.geometry)
        rows.append({
            "fire_id": fire["Cod_SGIF"], "municipality": fire["PI_Conc"],
            "area_ha": fire["AreaHaSIG"], "model": MODELS[m],
            "recall": recall, "n_evaluated_px": n_px,
        })
fire_results = pd.DataFrame(rows)
fire_results


### 7.1 Detection rate vs fire size (interactive)

In [ ]:
fig = px.scatter(
    fire_results, x="area_ha", y="recall", color="model", hover_data=["municipality", "fire_id"],
    log_x=True, range_y=(-0.02, 1.02),
    title="Per-event recall vs fire size (eroded ICNF polygons, >65 ha)",
    color_discrete_map={"EfficientNet-B2": "#2c7fb8", "Swin-YNet": "#d95f0e"},
)
fig.add_vline(x=SIZE_THRESHOLD_HA, line_dash="dot", line_color="grey",
               annotation_text="65 ha evaluable threshold", annotation_position="top")
fig.add_vline(x=HEADLINE_THRESHOLD_HA, line_dash="dash", line_color="black",
               annotation_text="945 ha headline threshold", annotation_position="top")
fig.update_layout(xaxis_title="fire area, eroded (ha, log scale)", yaxis_title="fraction of fire area detected")
fig.show()


### 7.2 Headline events (>945 ha)

In [ ]:
headline = fire_results[fire_results["area_ha"] > HEADLINE_THRESHOLD_HA].sort_values(
    ["area_ha", "model"], ascending=[False, True]
)
headline


## 8. Spatial error pattern (interactive)

Aggregate metrics say how much error there is; this says where. Each map blocks
the tile down by an 8x8 max-pool (so a block is flagged if any pixel in it is)
purely to make a 10980x6304 raster tractable to render, then colours each block
violet (model and records agree it is burned), red (model only, a likely false
alarm) or blue (records only, a miss). Clustering of red, especially near chip
edges, would point at the chip-boundary effect (`inference/chips.py` zero-pads
chips that fall over the tile edge); clustering of blue inside fire perimeters,
especially on the smaller end of the evaluable range, points at `MIN_PATCH_SIZE`
or `CLOSING_RADIUS` removing genuine small detections.


In [ ]:
def confusion_layers(model_kind):
    m = (obs[model_kind] == 1) & scored
    p = (burned[model_kind] == 1) & m
    t = REAL & m
    return (p & t), (p & ~t), (~p & t)   # TP, FP, FN

def pool_or(a, k=8):
    H, W = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    return a[:H, :W].reshape(H // k, k, W // k, k).max(axis=(1, 3))

def error_code(model_kind, k=8):
    tp, fp, fn = confusion_layers(model_kind)
    TP, FP, FN = pool_or(tp, k), pool_or(fp, k), pool_or(fn, k)
    code_arr = np.zeros(TP.shape, dtype=np.uint8)   # 0 = TN/not observed
    code_arr[FN] = 1
    code_arr[FP] = 2
    code_arr[TP] = 3
    return code_arr

COLORS = ["#f5f5f5", "#3273c4", "#d62728", "#8e44ad"]   # TN, FN(miss), FP(false alarm), TP(agree)
LABELS_ = ["no data / not burned", "records only (missed)", "model only (false alarm)", "agree: burned"]

fig = make_subplots(rows=1, cols=2, subplot_titles=list(MODELS.values()))
for i, m in enumerate(MODELS, start=1):
    fig.add_trace(
        go.Heatmap(z=error_code(m), colorscale=[[k / 3, c] for k, c in enumerate(COLORS)],
                   zmin=0, zmax=3, showscale=(i == 2),
                   colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=LABELS_)),
        row=1, col=i,
    )
    fig.update_yaxes(autorange="reversed", row=1, col=i, scaleanchor=f"x{i}" if i > 1 else "x")
fig.update_layout(title="Spatial agreement / disagreement with ICNF (8x8 block max-pool)", height=500)
fig.show()


## 9. What this means for hyperparameter tuning

Both adapters apply morphological post-processing after the network's own
per-pixel classification, with model-specific settings (`models/efficienT_b2_2classes/configs.py`,
`models/updated_model/bacdm_predict/AAA_Configs.py`):

| Setting | EfficientNet-B2 | Swin-YNet |
|---|---|---|
| `CLOSING_RADIUS` | 1 px | 3 px |
| `MIN_PATCH_SIZE` | 25 px (0.25 ha) | 25 px (0.25 ha) |
| probability threshold override | none (`CUTS_THRESHOLD=None`) | `CUTS_THRESHOLD=0.3`, but applied only to the unrelated `Cuts` class, not `Fires` |
| decision rule for the burned class | plain argmax, 2-class softmax | plain argmax against 4 other classes (`Background`, `ClearCuts`, `OtherCuts`, `AgriLoss`) |

Three implications follow directly from the results above, not from general
principle:

1. **`MIN_PATCH_SIZE` is set well below the median fire.** At 0.25 ha it removes
   any connected burned blob smaller than that, while the median ICNF fire in
   this tile is 1.98 ha and the bulk of the 182-event distribution sits under
   the 65 ha evaluable threshold used in section 7. If the per-event recall plot
   shows recall dropping off as fire size approaches the 65 ha floor (rather than
   staying flat down to the smallest evaluable fires), that is evidence
   `MIN_PATCH_SIZE` is already discarding genuine small-fire detections before
   they ever reach the accuracy assessment, and lowering it (at the cost of more
   speckle/false positives, since 0.25 ha is also roughly the size of stray
   misclassified pixels) would be the first thing to try, not a change to the
   network itself.
2. **`CLOSING_RADIUS` trades precision for recall by construction**: a larger
   structuring element bridges small gaps inside a detected burn scar (helping
   recall on fragmented detections) but also fattens the boundary of every
   detected blob outward (hurting precision, especially right at the eroded
   ground-truth edge this notebook scores against). Swin-YNet's radius is 3x
   EfficientNet-B2's. If section 5's results show Swin-YNet with higher recall
   but lower precision than EfficientNet-B2, `CLOSING_RADIUS` is a plausible
   partial explanation and a cheap one to test in isolation, by rerunning
   post-processing only (no re-inference needed) at radius 1 and re-scoring.
3. **Swin-YNet has no usable probability threshold for the burned class.**
   `CUTS_THRESHOLD` exists in the package and is already wired up for the
   `Cuts` class, but the burned class (`Fires`) still falls back to plain
   argmax against four competitor classes. Adding the same kind of threshold
   override for `Fires` (lower it to trade precision for recall, raise it to do
   the opposite) is a small, already-precedented code change in
   `bacdm_predict/predict.py`, and would let the team report a Swin-YNet
   precision/recall curve instead of a single operating point, which is
   currently only possible for EfficientNet-B2 in principle (its softmax
   probabilities are computed internally in `predict.py` but not exposed
   through the adapter either, so this applies to both models in practice and
   is the more fundamental gap if a true threshold sweep is wanted).


## 10. Limitations

- **No per-pixel cloud/QA mask.** `inference/chips.py` only screens NODATA
  (`observed = footprint & data != nodata`); scene-level cloud screening happens
  once per acquisition in `scene_select.py` (full_clear requires cloud <= 10%),
  but a `full_clear` scene can still contain isolated unmasked cloud or shadow
  pixels that neither model can distinguish from a genuine spectral change. This
  affects both models identically, so it should not bias the comparison between
  them, but it is a ceiling on the absolute accuracy either model can achieve
  here.
- **Large-N significance.** Section 6's McNemar test is run over tens of millions
  of pixels; treat the discordant-pixel count and which model wins on it as the
  meaningful number, not the p-value in isolation.
- **Single before/after pair.** Both models were run on the same one acquisition
  pair, so the per-acquisition diagnostic recommended in `evaluation_protocol.md`
  section 2.5 (does accuracy degrade with cloud fraction or temporal gap) is not
  exercised here. Re-running either model against a second after-date (the
  project's earlier exploratory notebook used both 2025-09-25 and 2025-10-15)
  would let that comparison happen, but is a separate, larger piece of work.
- **Erosion distance is a single fixed choice (10 m).** A larger or smaller
  erosion would tighten or loosen the boundary-ambiguity exclusion and shift
  every metric here slightly; it was not swept as a sensitivity check.


## References

Chicco, D. & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics*, 21, 6.

Chuvieco, E. et al. (2019). Historical background and current developments for mapping burned area from satellite Earth observation. *Remote Sensing of Environment*, 225, 45-64.

Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861-874.

Matthews, B.W. (1975). Comparison of the predicted and observed secondary structure of T4 phage lysozyme. *Biochimica et Biophysica Acta*, 405(2), 442-451.

Raschka, S., Liu, Y.H. & Mirjalili, V. (2022). *Machine Learning with PyTorch and Scikit-Learn*. Packt Publishing.

Roteta, E., Bastarrika, A., Padilla, M., Storm, T. & Chuvieco, E. (2019). Development of a Sentinel-2 burned area algorithm: Generation of a small fire database for sub-Saharan Africa. *Remote Sensing of Environment*, 222, 1-17.

Stehman, S.V. & Foody, G.M. (2019). Key issues in rigorous accuracy assessment of land cover products. *Remote Sensing of Environment*, 231, 111199.
